# 🎭 Real-Time AI Avatar Streaming Server
### RunPod Notebook — Port 7865

**Pipeline:**
```
Browser Mic → WebSocket → Moshi → Bridge → AvatarForcing → Browser (A+V)
```

**Architecture:** 7 concurrent async tasks per session:
- `audio_receive`  — browser mic PCM → audio_queue
- `moshi_loop`     — Moshi InferenceState → PCM send (priority) + tokens
- `bridge_loop`    — tokens → bridge (KV-cache) → audio_emb chunks
- `af_loop`        — audio_emb → AvatarForcing self-forcing → 4 frames/block
- `frame_encode`   — RGB → TurboJPEG → send_queue
- `keepalive`      — send_queue drain @ 25fps + 0xFE pings

**Port:** 7865 (RunPod proxied URL)


---
## 📦 Cell 1 — System Dependencies

In [ ]:
%%bash
apt-get update -qq && apt-get install -y -qq \
    ffmpeg libsndfile1 sox git wget curl \
    libturbojpeg-dev libjpeg-turbo8-dev 2>/dev/null
echo '✅ System deps installed (incl. libjpeg-turbo)'

## 📦 Cell 2 — Python: PyTorch

In [ ]:
!pip uninstall -y torch torchvision torchaudio flash-attn
!pip install torch==2.6.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
print('✅ PyTorch 2.6 + CUDA 12.4 installed')

## 📦 Cell 3 — Python: Pipeline Dependencies

In [ ]:
%%bash
pip install -q \
    "moshi==0.2.4" \
    "sphn>=0.1.4" \
    "sentencepiece>=0.1.99" \
    "transformers==4.41.2" \
    "accelerate==0.31.0" \
    "diffusers==0.30.0" \
    "peft==0.13.0" \
    "ftfy" \
    "easydict" \
    "safetensors>=0.4.0" \
    "huggingface_hub>=0.23.0" \
    "omegaconf>=2.3.0" \
    "einops>=0.7.0" \
    "soundfile>=0.12.1" \
    "pyyaml>=6.0" \
    "pillow>=10.0" \
    "decord>=0.6.0" \
    "lmdb>=1.4.0" \
    "pandas>=2.0" \
    "opencv-python-headless>=4.8" \
    "pyworld>=0.3.4" \
    "tqdm>=4.66" \
    "ipywidgets>=8.0" \
    "numpy" \
    "scipy" \
    "imageio" \
    "imageio-ffmpeg" \
    "av" \
    "rotary-embedding-torch" \
    "pytorch-lightning" \
    "torchmetrics"
echo '✅ Pipeline deps installed'

## 📦 Cell 4 — Python: Streaming Server Dependencies

In [ ]:
%%bash
pip install -q \
    "fastapi>=0.110.0" \
    "uvicorn[standard]>=0.29.0" \
    "websockets>=12.0" \
    "python-multipart>=0.0.9" \
    "PyTurboJPEG>=1.7.0"
echo '✅ Streaming server deps installed'

## 📦 Cell 5 — Flash Attention

In [ ]:
!pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl --no-build-isolation

In [ ]:
import flash_attn; print(f'✅ flash_attn {flash_attn.__version__}')

---
## 📂 Cell 6 — Workspace Setup

In [ ]:
import os, sys

WORKSPACE = '/workspace/Moshi-AvatarForcing-bridge-v4'

# If not already cloned:
if not os.path.exists(WORKSPACE):
    !git clone https://github.com/MoshiHead/Moshi-AvatarForcing-bridge-v4.git {WORKSPACE}

%cd {WORKSPACE}

# Add all roots to sys.path
MOSHI_ROOT  = f'{WORKSPACE}/moshi-inference'
BRIDGE_ROOT = f'{WORKSPACE}/moshi-wav2vec-bridge'
AF_ROOT     = f'{WORKSPACE}/AvatarForcing-inference'

for p in [WORKSPACE, MOSHI_ROOT, BRIDGE_ROOT, AF_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'✅ Workspace: {WORKSPACE}')
print(f'  Moshi:    {MOSHI_ROOT}')
print(f'  Bridge:   {BRIDGE_ROOT}')
print(f'  AF:       {AF_ROOT}')

---
## 📥 Cell 7 — Download Model Weights

In [ ]:
import os
from huggingface_hub import snapshot_download, hf_hub_download

os.makedirs('./checkpoints', exist_ok=True)

# ── Wan2.1-T2V-1.3B backbone ──────────────────────────────────────────────────
print('Downloading Wan2.1-T2V-1.3B …')
snapshot_download(
    repo_id   = 'Wan-AI/Wan2.1-T2V-1.3B',
    local_dir = './AvatarForcing-inference/wan_models/Wan2.1-T2V-1.3B',
    ignore_patterns = ['*.md', '*.txt'],
)
print('✅ Wan2.1-T2V-1.3B ready')

In [ ]:
# ── AvatarForcing checkpoint ──────────────────────────────────────────────────
from huggingface_hub import snapshot_download
print('Downloading AvatarForcing checkpoints …')
snapshot_download(
    repo_id=  'lycui/AvatarForcing',
    local_dir='./checkpoints',
    local_dir_use_symlinks=False,
)
print('✅ AvatarForcing checkpoint ready')

In [ ]:
# ── Bridge checkpoint ─────────────────────────────────────────────────────────
from huggingface_hub import hf_hub_download
hf_hub_download(
    repo_id   = 'Darknsu/new-bridge-model-12-layer-unified-train-v1',
    filename  = 'bridge_best.pt',
    repo_type = 'dataset',
    local_dir = './checkpoints',
)
BRIDGE_CKPT = './checkpoints/bridge_best.pt'
AF_CKPT     = './checkpoints/model.pt'
print(f'✅ Bridge: {BRIDGE_CKPT}  exists={os.path.exists(BRIDGE_CKPT)}')
print(f'✅ AF:     {AF_CKPT}      exists={os.path.exists(AF_CKPT)}')

---
## ⚙️ Cell 8 — Configure Paths

In [ ]:
import os

WORKSPACE = '/workspace/Moshi-AvatarForcing-bridge-v4'

# ── Set environment variables for the streaming server ───────────────────────
os.environ['BRIDGE_CKPT']   = f'{WORKSPACE}/checkpoints/bridge_best.pt'
os.environ['BRIDGE_CONFIG'] = f'{WORKSPACE}/moshi-wav2vec-bridge/config.yaml'
os.environ['AF_CKPT']       = f'{WORKSPACE}/checkpoints/model.pt'
os.environ['AF_CONFIG']     = f'{WORKSPACE}/AvatarForcing-inference/configs/avatarforcing.yaml'
os.environ['MOSHI_HF_REPO'] = 'kyutai/moshiko-pytorch-q8'
os.environ['DEVICE']        = 'cuda'
os.environ['PORT']          = '7865'
os.environ['HOST']          = '0.0.0.0'
os.environ['TEACHER_LEN']   = '80'
os.environ['USE_EMA']       = 'false'
os.environ['AF_PROMPT']     = (
    'A person talking naturally, realistic facial expressions, '
    'high quality video, detailed face.'
)

print('Configuration:')
for k in ['BRIDGE_CKPT','AF_CKPT','AF_CONFIG','DEVICE','PORT']:
    print(f'  {k} = {os.environ[k]}')

---
## 🔍 Cell 9 — Verify Imports

In [ ]:
import sys, os
WORKSPACE = '/workspace/Moshi-AvatarForcing-bridge-v4'
for p in [
    WORKSPACE,
    f'{WORKSPACE}/moshi-inference',
    f'{WORKSPACE}/moshi-wav2vec-bridge',
    f'{WORKSPACE}/AvatarForcing-inference',
]:
    if p not in sys.path: sys.path.insert(0, p)

# Verify core imports work
import torch
print(f'✅ torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from fastapi import FastAPI; print('✅ fastapi')
import uvicorn;               print('✅ uvicorn')
from omegaconf import OmegaConf; print('✅ omegaconf')
from einops import rearrange; print('✅ einops')

try:
    from turbojpeg import TurboJPEG; tj = TurboJPEG(); print('✅ TurboJPEG')
except Exception as e:
    print(f'⚠️  TurboJPEG not available ({e}), will use cv2 fallback')

print('\n✅ All imports verified.')

---
## 🚀 Cell 10 — Launch Streaming Server (Port 7865)

> **RunPod URL:** After running this cell, open:
> `https://<pod-id>-7865.proxy.runpod.net`
>
> ⚠️ The server blocks this cell while running. Use **Run All Above** first,
> then run this cell and switch to the browser tab.

In [ ]:
import subprocess, os, sys

WORKSPACE = '/workspace/Moshi-AvatarForcing-bridge-v4'
os.chdir(WORKSPACE)

# Set env for the server
env = os.environ.copy()

# ── Launch uvicorn directly (non-blocking in notebook) ───────────────────────
import asyncio
import uvicorn

# Add all paths
for p in [
    WORKSPACE,
    f'{WORKSPACE}/moshi-inference',
    f'{WORKSPACE}/moshi-wav2vec-bridge',
    f'{WORKSPACE}/AvatarForcing-inference',
]:
    if p not in sys.path: sys.path.insert(0, p)

print('=' * 60)
print('  Starting Real-Time AI Avatar Streaming Server')
print(f'  Port: 7865')
print('  Loading models (may take 1-3 min) …')
print('=' * 60)

# This call blocks — models load in the lifespan, then server accepts connections
uvicorn.run(
    'streaming_server.server:app',
    host              = '0.0.0.0',
    port              = 7865,
    log_level         = 'info',
    ws_ping_interval  = 20,
    ws_ping_timeout   = 60,
    timeout_keep_alive= 30,
)

---
## 🌐 Cell 11 — Get RunPod Public URL

In [ ]:
# Run this in a separate cell BEFORE launching the server
# to see your public URL
import os
pod_id = os.environ.get('RUNPOD_POD_ID', 'YOUR_POD_ID')
print(f'Public URL:  https://{pod_id}-7865.proxy.runpod.net')
print(f'Health check: https://{pod_id}-7865.proxy.runpod.net/health')
print()
print('Workflow:')
print('  1. Open the public URL in browser')
print('  2. Upload a frontal face image')
print('  3. Click "Start Session"')
print('  4. Click "Start Talking" and speak into your microphone')
print('  5. Watch the avatar respond in real-time')

---
## 🔧 Cell 12 — Quick Health Check

In [ ]:
# Run while server is running (in a separate kernel or after background launch)
import requests, json
try:
    r = requests.get('http://localhost:7865/health', timeout=5)
    print(json.dumps(r.json(), indent=2))
except Exception as e:
    print(f'Server not reachable: {e}')
    print('Run Cell 10 first.')

---
## 🔎 Cell 13 — Diagnostics: Bridge Shape Check

In [ ]:
# Verify bridge output shape matches AvatarForcing expectations
import torch, sys
WORKSPACE   = '/workspace/Moshi-AvatarForcing-bridge-v4'
BRIDGE_ROOT = f'{WORKSPACE}/moshi-wav2vec-bridge'
if BRIDGE_ROOT not in sys.path: sys.path.insert(0, BRIDGE_ROOT)

from streaming_server.pipeline.streaming_bridge import StreamingBridge
from model import MimiWav2Vec2Bridge
import yaml

with open(f'{BRIDGE_ROOT}/config.yaml') as f:
    cfg = yaml.safe_load(f)

model = MimiWav2Vec2Bridge(cfg).to('cuda')
ckpt  = torch.load(f'{WORKSPACE}/checkpoints/bridge_best.pt', map_location='cuda', weights_only=False)
sd    = ckpt.get('bridge', ckpt)
model.load_state_dict(sd, strict=False)
model.eval()

bridge = StreamingBridge(model, device=torch.device('cuda'))
bridge.reset_session()

# Simulate 4 tokens (2 flushes of BRIDGE_CHUNK=2)
result = None
for i in range(4):
    r = bridge.push_token(torch.randint(0, 2048, (1, 8)), seq=i)
    if r is not None:
        seq_s, seq_e, emb = r
        print(f'  Flush [{seq_s},{seq_e}): emb.shape={emb.shape}')
        assert emb.shape == (4, 10752), f'Expected (4, 10752), got {emb.shape}'

print('✅ Bridge streaming shape verified: (2*BRIDGE_CHUNK=4, 10752)')

---
## 📐 Architecture Reference

| Stage | Input | Output | Rate |
|---|---|---|---|
| Browser AudioWorklet | Microphone | int16 PCM chunks | 24kHz |
| Moshi Mimi Encode | PCM (1920 samples) | 8 codebook tokens | 12.5 Hz |
| Moshi Helium LM | 8 user tokens | 8 response tokens + 1 text | 12.5 Hz |
| Moshi Mimi Decode | 8 response tokens | PCM (1920 samples) | 24kHz |
| **Bridge (streaming, KV-cache)** | 2 tokens/flush | (4, 10752) audio_emb | 25 Hz |
| **AvatarForcing (self-forcing)** | audio_emb + image | 4 latent frames | 25 FPS |
| VAE Decode | 4 latent frames | 4 RGB frames (480×832) | 25 FPS |
| TurboJPEG | RGB frame | JPEG bytes | 25 FPS |
| WebSocket → Browser | JPEG + PCM | Synchronized A+V | real-time |

**Streaming Design Choices:**
- `inference_self_forcing` (NOT windowed forcing) — processes 1 block = 4 frames at a time
- KV-cache persists across ALL blocks in a session — no full-sequence recomputation
- Audio_emb padded with zeros if bridge hasn't produced enough yet — no startup delay
- Audio sent via priority write-lock bypass — video never blocks audio
- Frame queue bounded at 30 — oldest dropped under GPU overload